# 3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_olivetti_faces
from sklearn.preprocessing import label_binarize

roll = 26017
rng = np.random.RandomState(roll)
faces = fetch_olivetti_faces()
X = faces.data
y = faces.target
images = faces.images
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=rng)
print(f"Data shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"images shape: {images.shape}")

Data shape: (400, 4096)
Target shape: (400,)
images shape: (400, 64, 64)


In [ ]:
train_idx, val_idx, test_idx = [], [], []
for i in range(40):

    identity_indices = np.where(y == i)[0]
    rng.shuffle(identity_indices)
    train_idx.extend(identity_indices[:6])
    val_idx.extend(identity_indices[6:8])
    test_idx.extend(identity_indices[8:])

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test_clean, y_test = X[test_idx], y[test_idx]

X_test_masked = X_test_clean.copy()
for i in range(len(X_test_masked)):
    img = X_test_masked[i].reshape(64, 64)
    img[24:40, 24:40] = 0.0
    X_test_masked[i] = img.flatten()

print(f"Train size: {X_train.shape}, Val size: {X_val.shape}, Test size: {X_test_clean.shape}")

K = 40
N_test = len(y_test)
R = 30

def resample_within_identity(X, y, rng_seed):
    rng = np.random.default_rng(rng_seed)
    X_resampled, y_resampled = [], []
    for i in range(K):

        idx = np.where(y == i)[0]
        resampled_idx = rng.choice(idx, size=len(idx), replace=True)
        X_resampled.extend(X[resampled_idx])
        y_resampled.extend(y[resampled_idx])
    return np.array(X_resampled), np.array(y_resampled)

def compute_bias_variance(p_r_matrix, y_true):

    e_i = label_binarize(y_true, classes=np.arange(K))
    p_bar = np.mean(p_r_matrix, axis=0)
    bias_squared = np.mean(np.sum((p_bar - e_i)**2, axis=1))

    variance = np.mean(np.sum((p_r_matrix - p_bar)**2, axis=2))

    return bias_squared, variance

Train size: (240, 4096), Val size: (80, 4096), Test size: (80, 4096)


In [ ]:
max_depths = [2, 4, 8, 16, None]
min_samples_leafs = [1, 2, 4]

all_results = []

best_val_acc = -1
best_params = {}
best_tree = None

for md in max_depths:
    for msl in min_samples_leafs:
        dt = DecisionTreeClassifier(max_depth=md, min_samples_leaf=msl, random_state=rng)
        dt.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, dt.predict(X_train))
        val_acc = accuracy_score(y_val, dt.predict(X_val))
        test_acc = accuracy_score(y_test, dt.predict(X_test_clean))

        n_nodes = dt.tree_.node_count
        children_left = dt.tree_.children_left
        children_right = dt.tree_.children_right
        node_samples = dt.tree_.n_node_samples

        leaf_samples = []
        for i in range(n_nodes):
            if children_left[i] == children_right[i]: # Leaf node
                leaf_samples.append(node_samples[i])

        n_leaves = len(leaf_samples)
        mean_samples = np.mean(leaf_samples) if n_leaves > 0 else 0
        fraction_single = (sum(np.array(leaf_samples) == 1) / n_leaves) if n_leaves > 0 else 0

        all_results.append({
            "Max Depth": str(md),
            "Min Samples Leaf": msl,
            "Train Acc": f"{train_acc * 100:.2f}%",
            "Val Acc": f"{val_acc * 100:.2f}%",
            "Test Acc": f"{test_acc * 100:.2f}%",
            "Number of Leaves": n_leaves,
            "Mean Samples/Leaf": f"{mean_samples:.2f}",
            "Fraction Single-Sample Leaves": f"{fraction_single * 100:.2f}%"
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_params = {'max_depth': md, 'min_samples_leaf': msl}
            best_tree = dt

df_grid_results = pd.DataFrame(all_results)
display(df_grid_results)

,Max Depth,Min Samples Leaf,Train Acc,Val Acc,Test Acc,Number of Leaves,Mean Samples/Leaf,Fraction Single-Sample Leaves
0,2,1,7.50%,6.25%,6.25%,3,80.00,0.00%
1,2,2,7.50%,6.25%,5.00%,3,80.00,0.00%
2,2,4,7.50%,6.25%,7.50%,3,80.00,0.00%
3,4,1,12.50%,10.00%,11.25%,5,48.00,0.00%
4,4,2,12.50%,8.75%,7.50%,5,48.00,0.00%
5,4,4,12.50%,8.75%,10.00%,5,48.00,0.00%
6,8,1,25.00%,23.75%,22.50%,10,24.00,0.00%
7,8,2,22.50%,16.25%,16.25%,9,26.67,0.00%
8,8,4,22.50%,17.50%,20.00%,9,26.67,0.00%
9,16,1,45.00%,33.75%,35.00%,18,13.33,0.00%


In [ ]:
depths = [1, 2, 3, 4]
rounds = [10, 25, 50, 100]

def train_samme(X_tr, y_tr, max_depth, n_rounds):
    N = len(X_tr)
    w = np.ones(N) / N
    models, alphas = [], []

    err_history, alpha_history, max_w_history, top10_w_history = [], [], [], []

    for t in range(n_rounds):
        dt = DecisionTreeClassifier(max_depth=max_depth, random_state=t)
        dt.fit(X_tr, y_tr, sample_weight=w)
        preds = dt.predict(X_tr)

        mistakes = (preds != y_tr)
        err_t = np.sum(w[mistakes])

        if err_t >= (1.0 - 1.0 / K):
            break

        if err_t == 0:
            alpha_t = np.log((1 - 1e-10) / 1e-10) + np.log(K - 1)
        else:
            alpha_t = np.log((1 - err_t) / err_t) + np.log(K - 1)

        w_unnorm = w * np.exp(alpha_t * mistakes)
        w = w_unnorm / np.sum(w_unnorm)

        models.append(dt)
        alphas.append(alpha_t)

        err_history.append(err_t)
        alpha_history.append(alpha_t)
        max_w_history.append(np.max(w))
        top10_w_history.append(np.sum(np.sort(w)[-10:]))

    return models, alphas, w, err_history, alpha_history, max_w_history, top10_w_history

def predict_samme(models, alphas, X_data):

    n_samples = len(X_data)
    scores = np.zeros((n_samples, K))

    for dt, alpha in zip(models, alphas):
        preds = dt.predict(X_data)
        for i in range(n_samples):
            scores[i, preds[i]] += alpha

    final_preds = np.argmax(scores, axis=1)

    confidences = np.max(scores, axis=1) / np.sum(alphas)
    return final_preds, confidences

best_val_acc = -1
best_samme_model = None

for d in depths:
    for r in rounds:
        models, alphas, _, _, _, _, _ = train_samme(X_train, y_train, d, r)
        val_preds, _ = predict_samme(models, alphas, X_val)
        val_acc = np.mean(val_preds == y_val)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_samme_model = (d, r)

print(f"\nModel Selection:")
print(f"Best SAMME parameters (depth, rounds): {best_samme_model} with Val Acc: {best_val_acc:.4f}")

best_d, best_r = best_samme_model
final_models, final_alphas, final_weights, err_hist, alpha_hist, max_w_hist, top10_w_hist = train_samme(X_train, y_train, best_d, best_r)



Model Selection:
Best SAMME parameters (depth, rounds): (4, 50) with Val Acc: 0.4250


In [ ]:
#3.(a)
tree_p_r = np.zeros((R, N_test, K))
bag_p_r = np.zeros((R, N_test, K))
boost_p_r = np.zeros((R, N_test, K))

tree_accs, bag_accs, boost_accs = [], [], []
tree_depth = best_params['max_depth']
bag_B = 100
boost_d, boost_r = best_samme_model

for r in range(R):
    Xr, yr = resample_within_identity(X_train, y_train, rng_seed=r)

    dt = DecisionTreeClassifier(**best_params, random_state=r)
    dt.fit(Xr, yr)
    preds = dt.predict(X_test_clean)
    tree_accs.append(np.mean(preds == y_test))
    tree_p_r[r] = label_binarize(preds, classes=np.arange(K))

    bag_models = []
    for b in range(bag_B):

        boot_idx = np.random.choice(len(Xr), size=len(Xr), replace=True)
        dt_bag = DecisionTreeClassifier(**best_params, random_state=b*R + r)
        dt_bag.fit(Xr[boot_idx], yr[boot_idx])
        bag_models.append(dt_bag)

    bag_preds_raw = np.array([m.predict(X_test_clean) for m in bag_models])
    bag_maj_preds = np.array([np.bincount(bag_preds_raw[:, i], minlength=K).argmax() for i in range(N_test)])
    bag_accs.append(np.mean(bag_maj_preds == y_test))
    bag_p_r[r] = label_binarize(bag_maj_preds, classes=np.arange(K))

    b_models, b_alphas, _, _, _, _, _ = train_samme(Xr, yr, boost_d, boost_r)
    boost_preds, boost_conf = predict_samme(b_models, b_alphas, X_test_clean)
    boost_accs.append(np.mean(boost_preds == y_test))

    scores = np.zeros((N_test, K))
    for model, alpha in zip(b_models, b_alphas):
        p = model.predict(X_test_clean)
        for i in range(N_test):
            scores[i, p[i]] += alpha
    boost_p_r[r] = scores / np.sum(b_alphas)

tree_bias_sq, tree_var = compute_bias_variance(tree_p_r, y_test)
bag_bias_sq, bag_var = compute_bias_variance(bag_p_r, y_test)
boost_bias_sq, boost_var = compute_bias_variance(boost_p_r, y_test)

print(f"Single Tree - Bias-square: {tree_bias_sq:.4f}, Variance: {tree_var:.4f}, Acc: {np.mean(tree_accs):.4f} +/- {np.std(tree_accs):.4f}")
print(f"Bagging     - Bias-square: {bag_bias_sq:.4f}, Variance: {bag_var:.4f}, Acc: {np.mean(bag_accs):.4f} +/- {np.std(bag_accs):.4f}")
print(f"Boosting    - Bias-square: {boost_bias_sq:.4f}, Variance: {boost_var:.4f}, Acc: {np.mean(boost_accs):.4f} +/- {np.std(boost_accs):.4f}")

Single Tree - Bias-square: 0.3774, Variance: 0.6484, Acc: 0.4871 +/- 0.0452
Bagging     - Bias-square: 0.1609, Variance: 0.2700, Acc: 0.7846 +/- 0.0417
Boosting    - Bias-square: 0.8835, Variance: 0.0206, Acc: 0.2725 +/- 0.0600


In [ ]:
#3.(b)
depths_to_test = [2, tree_depth, None]
results_b = {}

for d in depths_to_test:
    t_pr_d = np.zeros((R, N_test, K))
    b_pr_d = np.zeros((R, N_test, K))

    for r in range(R):
        Xr, yr = resample_within_identity(X_train, y_train, rng_seed=r)

        dt = DecisionTreeClassifier(max_depth=d, random_state=r)
        dt.fit(Xr, yr)
        t_pr_d[r] = label_binarize(dt.predict(X_test_clean), classes=np.arange(K))

        bag_models = []
        for b in range(100):
            boot_idx = np.random.choice(len(Xr), size=len(Xr), replace=True)
            dt_bag = DecisionTreeClassifier(max_depth=d, random_state=b*R + r)
            dt_bag.fit(Xr[boot_idx], yr[boot_idx])
            bag_models.append(dt_bag)

        bag_preds_raw = np.array([m.predict(X_test_clean) for m in bag_models])
        bag_maj_preds = np.array([np.bincount(bag_preds_raw[:, i], minlength=K).argmax() for i in range(N_test)])
        b_pr_d[r] = label_binarize(bag_maj_preds, classes=np.arange(K))

    tb_sq, tvar = compute_bias_variance(t_pr_d, y_test)
    bb_sq, bvar = compute_bias_variance(b_pr_d, y_test)

    variance_removed = tvar - bvar
    results_b[d] = {'tree_bias-square': tb_sq, 'tree_var': tvar, 'bag_bias-square': bb_sq, 'bag_var': bvar, 'var_removed': variance_removed}
    print(f"Depth {d}: Tree Bias-square={tb_sq:.4f}, Tree Var={tvar:.4f} | Bag Bias-square={bb_sq:.4f}, Bag Var={bvar:.4f} | Var Removed={variance_removed:.4f}")

Depth 2: Tree Bias-square=1.7976, Tree Var=0.0857 | Bag Bias-square=0.8759, Bag Var=0.8541 | Var Removed=-0.7684
Depth None: Tree Bias-square=0.3774, Tree Var=0.6484 | Bag Bias-square=0.1402, Bag Var=0.2590 | Var Removed=0.3895
Depth None: Tree Bias-square=0.3774, Tree Var=0.6484 | Bag Bias-square=0.1482, Bag Var=0.2585 | Var Removed=0.3900


In [ ]:
#3.(c)
# Single
dt_selected = DecisionTreeClassifier(**best_params, random_state=42)
dt_selected.fit(X_train, y_train)
tree_mask_preds = dt_selected.predict(X_test_masked)
tree_mask_acc = np.mean(tree_mask_preds == y_test)

# Bagging
bag_models_selected = []
for b in range(100):
    boot_idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
    dt_bag = DecisionTreeClassifier(**best_params, random_state=b)
    dt_bag.fit(X_train[boot_idx], y_train[boot_idx])
    bag_models_selected.append(dt_bag)

bag_preds_raw = np.array([m.predict(X_test_masked) for m in bag_models_selected])
bag_mask_preds = np.array([np.bincount(bag_preds_raw[:, i], minlength=K).argmax() for i in range(N_test)])
bag_mask_acc = np.mean(bag_mask_preds == y_test)

# Boosting
boost_d, boost_r = best_samme_model
b_models_selected, b_alphas_selected, _, _, _, _, _ = train_samme(X_train, y_train, boost_d, boost_r)
boost_mask_preds, _ = predict_samme(b_models_selected, b_alphas_selected, X_test_masked)
boost_mask_acc = np.mean(boost_mask_preds == y_test)

data = {
    "Model": ["Single Tree (Selected)", "Bagging (Selected)", "Boosting (Selected)"],
    "Clean Test Acc (Mean)": [np.mean(tree_accs), np.mean(bag_accs), np.mean(boost_accs)],
    "Masked Test Acc": [tree_mask_acc, bag_mask_acc, boost_mask_acc],
    "Measured Bias^2": [tree_bias_sq, bag_bias_sq, boost_bias_sq],
    "Measured Variance": [tree_var, bag_var, boost_var]
}

df_results = pd.DataFrame(data)
print("\nFinal Bias, Variance, and Accuracy Table:")
print(df_results.to_string(index=False))


Final Bias, Variance, and Accuracy Table:
                 Model  Clean Test Acc (Mean)  Masked Test Acc  Measured Bias^2  Measured Variance
Single Tree (Selected)               0.487083           0.3125         0.377389           0.648444
    Bagging (Selected)               0.776667           0.8250         0.167833           0.278833
   Boosting (Selected)               0.272500           0.3625         0.883492           0.020592


In [ ]:
import numpy as np

# Prepare structured outputs for each section to be saved in the npz file
section_3a = {
    'tree_accs': tree_accs,
    'bag_accs': bag_accs,
    'boost_accs': boost_accs,
    'tree_bias_sq': tree_bias_sq,
    'tree_var': tree_var,
    'bag_bias_sq': bag_bias_sq,
    'bag_var': bag_var,
    'boost_bias_sq': boost_bias_sq,
    'boost_var': boost_var
}

# For 3.(b), we save the dictionary of dictionary results by converting or directly saving its content
section_3b = {
    'depths_tested': np.array(list(results_b.keys()), dtype=object),
    'results': np.array(list(results_b.values()))
}

section_3c = {
    'tree_mask_acc': tree_mask_acc,
    'bag_mask_acc': bag_mask_acc,
    'boost_mask_acc': boost_mask_acc
}

# 3.(d) represents the consolidated summary/table data
section_3d = {
    'summary_table': df_results.to_csv(index=False)  # Saving table as a CSV string representation
}

# Save the results partitioned by section headings in the npz file
np.savez(
    'resultsQ1.npz',
    **{
        '3a_tree_accs': section_3a['tree_accs'],
        '3a_bag_accs': section_3a['bag_accs'],
        '3a_boost_accs': section_3a['boost_accs'],
        '3a_tree_bias_sq': section_3a['tree_bias_sq'],
        '3a_tree_var': section_3a['tree_var'],
        '3a_bag_bias_sq': section_3a['bag_bias_sq'],
        '3a_bag_var': section_3a['bag_var'],
        '3a_boost_bias_sq': section_3a['boost_bias_sq'],
        '3a_boost_var': section_3a['boost_var'],
        '3b_depths_tested': section_3b['depths_tested'],
        '3b_results': section_3b['results'],
        '3c_tree_mask_acc': section_3c['tree_mask_acc'],
        '3c_bag_mask_acc': section_3c['bag_mask_acc'],
        '3c_boost_mask_acc': section_3c['boost_mask_acc'],
        '3d_summary_table': section_3d['summary_table']
    }
)

print("Results successfully saved to resultsQ1.npz with separate sections for 3.(a), 3.(b), 3.(c), and 3.(d)!")